# 02 - Feature Extraction

This notebook extracts Concerto Small features for one selected S3DIS area. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data.

### 1. Setup & Mount Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/eval-demo'
AREA_ID = 4
AREA_NAME = f'Area_{AREA_ID}'
FEATURES_NAME = f's3dis_area{AREA_ID}'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

/content/Deep_learning_project
Branch 'dev/eval-demo' set up to track remote branch 'dev/eval-demo' from 'origin'.
Reset branch 'dev/eval-demo'
Your branch is up to date with 'origin/dev/eval-demo'.
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/eval-demo -> FETCH_HEAD
Already up to date.
dev/eval-demo
b4ae5e7 (HEAD -> dev/eval-demo, origin/dev/eval-demo) setub eval


In [10]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto

fatal: destination path '/content/Concerto' already exists and is not an empty directory.


### 2. Symlink Data & Checkpoints
We map the Drive folders directly into the repository so our scripts can find them.

In [11]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

In [12]:
%cd notebooks/

/content/Deep_learning_project/notebooks


### 3. Setup Hugging Face Token
Input your Hugging Face token below to allow `open_clip_torch` to download the models.

In [13]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. Run Feature Extraction
This script will iterate over the selected S3DIS area and extract the features. It is resume-safe, so if Colab disconnects, you can just run it again.

In [14]:
# Forzar a uv a usar/instalar Python 3.10.12 y sincronizar
!uv python install 3.10.12
!uv sync --python 3.10.12

Installed Python 3.10.12 in 1.05s
 + cpython-3.10.12-linux-x86_64-gnu (python3.10)
Using CPython 3.10.12
Creating virtual environment at: .venv
Resolved 147 packages in 4.33s
Prepared 140 packages in 1m 24s
Installed 141 packages in 1.62s
 + addict==2.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + blinker==1.9.0
 + brotli==1.2.0
 + camtools==0.1.8
 + ccimport==0.4.4
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + configargparse==1.7.5
 + contourpy==1.3.2
 + cumm-cu120==0.4.11
 + cycler==0.12.1
 + dash==4.1.0
 + dotenv==0.9.9
 + exceptiongroup==1.3.1
 + fast-pytorch-kmeans==0.2.2
 + fastapi==0.136.1
 + fastjsonschema==2.21.2
 + filelock==3.29.0
 + fire==0.7.1
 + flask==3.1.3
 + fonttools==4.62.1
 + fsspec==2026.4.0
 + ftfy==6.3.1
 + gradio==6.14.0
 + gradio-client==2.5.0
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.5.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.14.0
 + idna==3.13
 + imag

In [16]:
# Run the real extraction script with chunking to avoid T4 OOM on large rooms
print('Selected area:', AREA_NAME)
print('Output feature folder:', FEATURES_NAME)
!PYTHONPATH=/content/Concerto uv run python /content/Deep_learning_project/scripts/extract_features.py --data_dir /content/Deep_learning_project/data/s3dis_processed --out_dir /content/Deep_learning_project/features/{FEATURES_NAME} --areas {AREA_ID} --feature-dtype float16 --max-points-per-chunk 100000

Selected area: Area_4
Output feature folder: s3dis_area4
Loading S3DIS areas [4] from /content/Deep_learning_project/data/s3dis_processed...
S3DISDataset: 49 rooms loaded from areas [4]
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., d

auto: notebook execution

# Other tests

In [17]:
#!PYTHONPATH=/content/Concerto uv run python /content/Concerto/demo/0_pca.py

In [18]:
#!PYTHONPATH=/content/Concerto uv run python -c "import sys, torch; sys.path.insert(0, '/content/Concerto'); import concerto; import spconv.pytorch as spconv; print('python ok'); print(torch.__version__, torch.version.cuda); print(spconv.__file__)"